In [1]:
# variant level filter table
import pandas as pd

In [2]:
## columns
# keys: TBD

# disordered (plddt) (uni, aa pos, aa ref)
# ordered (plddt) (uni, aa pos, aa ref)
# buried (rSASA) (uni, aa pos, aa ref)
# surface (rSASA) (uni, aa pos, aa ref)

# destabilizing (foldX) (uni, aa pos, aa ref, aa alt) *** same question as AF2bind
# non destabilizing (foldX) (uni, aa pos, aa ref, aa alt) *** same question as AF2bind
# [aa property x4] (aa ref)
# af2bind (uni, aa pos, aa ref) *** ask about id/isoform issue - how careful do we want to be?
# mutation rate (no duplicates - we are good) *** check that I got the groups that we want
## top 10% (chrom, pos, ref, alt) - lowest mutation rate *** ask about this on Tuesday
## bottom 90% (chrom, pos, ref, alt) - highest mutation rate

In [3]:
# isoform information
df_p = pd.read_csv('../../scan_test_project/all_missense_variants_2_2026/protein_sequence_guide.tsv', sep='\t')
df_p = df_p[df_p.uniprot_canonical]
df_p = df_p[['uniprot_id', 'uniprot_isoform']].drop_duplicates()

In [4]:
# mutation rate
mu_df = pd.read_csv('../filter_files_v1/mutation_rate.tsv.bgz', sep='\t', compression='gzip')
# # deciles
# cutoff = mu_df["mu_snp"].quantile(0.10)
# mu_df["low_10_mu"] = mu_df["mu_snp"] <= cutoff
# mu_df["high_90_mu"] = mu_df["mu_snp"] > cutoff
# # change to all deciles **** 
# # look at histogram - is it continuous or categorical - might be categorical 
# # check in with Konrad and Jeremy later 
# df = df.merge(mu_df[['chrom', 'pos', 'ref', 'alt', 'low_10_mu', 'high_90_mu']], 
#               how='left', on=['chrom', 'pos', 'ref', 'alt'])

# assign deciles (0–9)
mu_df["mu_decile"] = pd.qcut(mu_df["mu_snp"], 10, labels=False)

# create boolean columns for each decile
decile_dummies = pd.get_dummies(mu_df["mu_decile"], prefix="mu_decile").astype(bool)

# combine back
mu_df = pd.concat([mu_df, decile_dummies], axis=1)

In [5]:
# af2bind
af2bind = pd.read_csv('../../reference_data/af2bind/expanded_pockets.tsv', sep='\t')
# get individual residues in pockets
af2bind=af2bind[['uniprot', 'expanded_pocket']]
def parse_int_list(s):
    # remove brackets and newlines, then split on whitespace
    return list(map(int, s.strip('[]').replace('\n', ' ').split()))

af2bind['expanded_pocket'] = af2bind['expanded_pocket'].apply(parse_int_list)
af2bind_vars = af2bind.explode('expanded_pocket')
af2bind_vars = af2bind_vars.drop_duplicates()
af2bind_vars = af2bind_vars.rename(columns={'uniprot': 'uniprot_id', 'expanded_pocket': 'aa_pos'})
af2bind_vars['predicted_pocket'] = True
# af2bind only gives ID not isoform - but should be based off of the canonical isoform?
af2bind_vars = af2bind_vars.merge(df_p, how='inner', on=['uniprot_id']) # gives us the canonical isoform (to the best of my ability)
af2bind_vars = af2bind_vars[["aa_pos", "predicted_pocket", "uniprot_isoform"]]

In [6]:
# aa property
# 'Nonpolar/Neutral'
np_aa = ['A', 'G', 'I', 'L', 'M', 'P', 'V']
# 'Polar/Neutral'
p_aa = ['C', 'N', 'Q', 'S', 'T']
# 'Charged'
c_aa = ['D', 'E', 'H', 'K', 'R']
# 'Aromatic'
a_aa = ['F', 'W', 'Y']

In [7]:
# foldX
df_f = pd.read_csv('../../reference_data/foldx/foldx_max_plddt_clean.tsv.gz', sep='\t', compression='gzip')
# add in isoform information - skipping for now - how comfortable are we with dropping data? or having questionable data?
df_f = df_f.merge(df_p, how='inner', on='uniprot_id')
#df_f = df_f[['aa_pos', 'aa_ref', 'aa_alt', 'uniprot_isoform', 'destabilizing']]
df_f = df_f[['aa_pos', 'aa_ref', 'aa_alt', 'uniprot_id', 'destabilizing']]
df_f['non_destabilizing'] = ~df_f.destabilizing

In [8]:
# pLDDT
df_plddt = pd.read_csv('../../reference_data/plddt/plddt_missense_variants_gr38.tsv.gz', sep='\t', compression='gzip')
# add in isoform information
df_plddt = df_plddt.merge(df_p, how='inner', on='uniprot_id')
df_plddt = df_plddt[['aa_pos', 'aa_ref', 'avg_plddt', 'uniprot_isoform', 'disordered']]
df_plddt['ordered'] = ~df_plddt.disordered

In [9]:
df_plddt.head()

,aa_pos,aa_ref,avg_plddt,uniprot_isoform,disordered,ordered
0,1,M,55.97,Q9Y2D2-1,True,False
1,2,F,64.81,Q9Y2D2-1,True,False
2,3,A,70.69,Q9Y2D2-1,False,True
3,4,N,85.31,Q9Y2D2-1,False,True
4,5,L,88.50,Q9Y2D2-1,False,True


In [10]:
# rSASA
df_rsasa = pd.read_csv('../../reference_data/rsasa/rsasa_missense_variants_gr38.tsv.gz', sep='\t', compression='gzip')
# add in isoform information
df_rsasa = df_rsasa.merge(df_p, how='inner', on='uniprot_id')
df_rsasa = df_rsasa[['aa_pos', 'aa_ref', 'uniprot_isoform', 'buried']]
df_rsasa['surface'] = ~df_rsasa['buried']

In [12]:
# CATH Class
feature = 'cath_class' # protein subset - general
ps_path = '../domains/uniprot_superfamily_clean.tsv' # protein subset data path; should be a tsv file
ps_col = 'CATH_class' # column of relevant data
ps_df = pd.read_csv(ps_path, sep='\t')
id_col = 'uniprot_id'
# create list of positions per row (inclusive range)
ps_df["aa_pos"] = ps_df.apply(lambda row: list(range(row["aa_start"], row["aa_end"] + 1)), axis=1)
ps_df['cath_class'] = ps_df.gene3d_id.str.split(':').str[1].str.split('.').str[0]
ps_df['cath_architecture'] = ps_df.gene3d_id.str.split(':').str[1].str.split('.').str[1]

# explode into separate rows
df_expanded = ps_df[["uniprot_isoform", "gene3d_id", "cath_class", "cath_architecture", "aa_pos"]].explode("aa_pos").reset_index(drop=True)

In [30]:
chromosome='6'
print('chromosome '+chromosome)
# this table (use one chrom for now)
df = pd.read_csv(f'gs://genetics-gym/linkers/linker_missense_enst_transcript_aa_uniprot_by_chrom_tsv/linker_missense_enst_transcript_aa_uniprot_{chromosome}.tsv.bgz',
                 sep='\t', compression='gzip')
df = df[['chrom', 'pos', 'ref', 'alt', 'ensg', 'enst', 'uniprot_isoform', 'uniprot_id', 'aa_pos', 'aa_ref', 'aa_alt']]

print('adding in mutation rate')
df = df.merge(
    mu_df[['chrom', 'pos', 'ref', 'alt'] + list(decile_dummies.columns)],
    how='left',
    on=['chrom', 'pos', 'ref', 'alt']
)

print('adding in af2bind')
df = df.merge(af2bind_vars, how='left', on=['uniprot_isoform', 'aa_pos'])

print('adding in amino acid properties')
df['nonpolar_neutral_aa'] = df.aa_ref.isin(np_aa)
df['polar_neutral_aa'] = df.aa_ref.isin(p_aa)
df['charged_aa'] = df.aa_ref.isin(c_aa)
df['aromatic_aa'] = df.aa_ref.isin(a_aa)

print('adding in foldx')
df = df.merge(df_f, how='left', on=['uniprot_id', 'aa_pos', 'aa_ref', 'aa_alt'])

print('adding in plddt')
df = df.merge(df_plddt, how='left', on=['uniprot_isoform', 'aa_pos', 'aa_ref'])

print('adding in rsasa')
df = df.merge(df_rsasa, how='left', on=['uniprot_isoform', 'aa_pos', 'aa_ref'])

print('adding in cath information')
df = df.merge(df_expanded, how='left', on=['uniprot_isoform', 'aa_pos'])

print('saving file')
df.to_csv(f'variant_filters_by_chrom/variant_filters_chr{chromosome}.tsv', sep='\t', index=False)

chromosome 6


/var/folders/13/9h9672rn4fq_js3d_7_vn7k40000gp/T/ipykernel_65377/751535114.py:4: DtypeWarning: Columns (0: transcript_mane_select) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f'gs://genetics-gym/linkers/linker_missense_enst_transcript_aa_uniprot_by_chrom_tsv/linker_missense_enst_transcript_aa_uniprot_{chromosome}.tsv.bgz',


adding in mutation rate
adding in af2bind
adding in amino acid properties
adding in foldx
adding in plddt
adding in rsasa
adding in cath information
saving file
